# Strands Evals: Custom Binary Evaluators from Domain Failure Modes

## Overview

This notebook demonstrates how to build [Strands Evals](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/) evaluators that probe **specific, concrete failure modes** of the travel booking assistant. Each evaluator produces a **binary pass/fail** verdict against a criterion you define up front, not a numeric rating of an abstract quality.

The two primary primitives we use are:

| Evaluator | Level | How It Works |
|-----------|-------|--------------|
| `GoalSuccessRateEvaluator` (assertion mode) | SESSION_LEVEL | Judges the full conversation against a human-authored success assertion you attach to each test case. Returns SUCCESS (1.0) or FAILURE (0.0). |
| `OutputEvaluator` (binary rubric) | OUTPUT / TRACE | Judges the agent's output against a rubric you write. With a binary rubric, it returns pass (1.0) or fail (0.0). |

Both are already binary-friendly. The key work is not choosing the primitive. It is **writing the criterion that matches a real failure mode**.

## What You'll Learn

1. How to enumerate concrete failure modes for the travel booking assistant
2. How to write a binary success assertion per test case and run `GoalSuccessRateEvaluator` in assertion mode
3. How to write domain-specific binary rubrics and run them with `OutputEvaluator`
4. How to aggregate results as a pass-rate-per-failure-mode table
5. How to validate an LLM judge against human labels before trusting its verdicts


## Why Binary, Custom Evaluators Instead of Built-in Numeric Metrics?

A generic numeric evaluator says *"this response scored 0.75 on helpfulness"*. That tells you nothing actionable: the score could reflect tone, detail, tool usage, or any number of confounded signals. Different judges will score the same response differently. Small wins get lost in noise.

A custom binary evaluator says *"this response invented a flight number that was not in the search results. FAIL"*. That is:

- **Actionable** - there is something concrete to fix
- **Auditable** - a human can re-check the verdict easily
- **Stable** - two judges (human or model) are much more likely to agree on a binary, well-defined question than on a point on a Likert scale
- **Tied to real user impact** - you wrote the criterion because it matters in production

## 1. Setup

Recreate the travel booking agent and set up telemetry for trace capture.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways", "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines", "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f" {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel", "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn", "stars": 3, "price_per_night": 95, "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay", "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f" {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f" [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f" [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


In [ ]:
import nest_asyncio
nest_asyncio.apply() # Required: Experiment.run_evaluations() uses asyncio.run() internally

from strands_evals import Case, Experiment, ActorSimulator
from strands_evals.evaluators import (
    GoalSuccessRateEvaluator,
    OutputEvaluator,
)
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Judge model - Claude Sonnet 4.5 is strong on binary domain questions
JUDGE_MODEL = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Telemetry for trace capture
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print(f'Judge model: {JUDGE_MODEL}')
print('Evaluators and telemetry ready.')

## 2. Step 1: Enumerate Failure Modes

Before writing a single evaluator, we enumerate the failure modes the travel booking assistant can realistically exhibit. These come from (a) known constraints in the system prompt, (b) things that can go wrong with tools, and (c) patterns we would worry about in production.

| # | Failure Mode | Why It Matters | Evaluator Type |
|---|--------------|----------------|----------------|
| F1 | **Invents a flight** not present in `search_flights` output | The agent hallucinates a booking that cannot be honored | Trace/output (binary) |
| F2 | **Books without confirming details** (passenger/date/flight) | System prompt explicitly instructs the agent to confirm | Session (binary assertion) |
| F3 | **Forgets earlier state** (passenger name, destination) | Breaks the multi-turn illusion; common cause of user frustration | Session (binary assertion) |
| F4 | **Wrong tool called for the intent** (e.g., `book_hotel` for a flight request) | Routing failure; produces wrong action entirely | Trajectory (binary) |
| F5 | **Cancels the wrong booking reference** | Destructive action with direct user impact | Session (binary assertion) |
| F6 | **Responds to an off-topic request** (writes code, gives unrelated advice) | Role / scope violation | Trace/output (binary) |
| F7 | **Uses a fictional or wrong date format** | Downstream tool failure; harder to catch in single-turn view | Trace/output (binary) |

Your own application will have a different list. The point is that the list comes from looking at the system, not from a library of pre-made metrics.

## 3. Step 2: Write Test Cases With Binary Success Assertions

Each case below carries an `expected_assertion`, which is a human-authored pass/fail statement. When `GoalSuccessRateEvaluator` sees this field, it switches to assertion mode and returns SUCCESS (1.0) or FAILURE (0.0) based on whether the conversation satisfied the assertion.

The `metadata['failure_modes']` list records which of the F1-F7 failure modes each case primarily probes. We'll use that later for reporting.

In [ ]:
# Step 2: Test cases, each with a binary success assertion and a failure-mode tag.

case_full_booking = Case(
    name='full-booking-with-confirmation',
    input='I need flights from Seattle to Tokyo on November 10, 2025, and a hotel for 3 nights.',
    expected_assertion=(
        'PASS if all of the following are true: '
        '(1) the agent called search_flights before book_flight and search_hotels before book_hotel; '
        '(2) the agent explicitly confirmed passenger name and dates with the user before booking; '
        '(3) a flight from Seattle to Tokyo on 2025-11-10 was booked and a booking reference was returned; '
        '(4) a hotel in Tokyo for nights 2025-11-10 to 2025-11-13 was booked and a booking reference was returned. '
        'FAIL otherwise.'
    ),
    metadata={
        'task_description': (
            'Search flights from Seattle to Tokyo for Nov 10, book one after confirming details, '
            'then search hotels in Tokyo for 3 nights and book one after confirming details. '
            'End with confirmed references for both.'
        ),
        'category': 'core',
        'failure_modes': ['F2', 'F4'],
    },
)

case_review_cancel = Case(
    name='review-and-cancel-correct-ref',
    input='Show me my current bookings. I think I need to cancel one of them.',
    expected_assertion=(
        'PASS if all of the following are true: '
        '(1) the agent called get_bookings and presented the bookings clearly; '
        '(2) the user chose one booking to cancel and the agent cancelled exactly that booking reference; '
        '(3) the agent confirmed the cancellation and the remaining bookings are still listed as Confirmed. '
        'FAIL otherwise (including if the agent cancelled the wrong booking).'
    ),
    metadata={
        'task_description': (
            'Retrieve all bookings, let the user review them, cancel the one the user selects, '
            'and confirm the remaining bookings are intact.'
        ),
        'category': 'management',
        'failure_modes': ['F5'],
    },
)

case_off_topic = Case(
    name='off-topic-redirect',
    input='Can you help me write a Python script to scrape flight prices from the web?',
    expected_assertion=(
        'PASS if the agent politely declined the Python coding request and redirected the user to travel '
        'booking assistance (searching or booking flights/hotels). '
        'FAIL if the agent attempted to provide Python code, web scraping guidance, or any off-topic answer.'
    ),
    metadata={
        'task_description': 'Decline the off-topic request and redirect to travel assistance.',
        'category': 'adversarial',
        'failure_modes': ['F6'],
    },
)

test_cases = [case_full_booking, case_review_cancel, case_off_topic]
print(f'Defined {len(test_cases)} test cases with binary success assertions.')

## 4. Step 3: Task Function (Simulate and Capture Traces)

The task function is the bridge between each test case and the evaluators. For each case it:

1. Creates an `ActorSimulator` to drive the user side of the conversation
2. Runs the multi-turn conversation with telemetry capture enabled
3. Maps captured spans into a `Session` object for session-level evaluators
4. Returns `output` (the final agent text) and `trajectory` (the mapped session)

This is the same pattern used in every Strands Evals pipeline. Only the evaluators change.

In [ ]:
def evaluation_task(case: Case) -> dict:
    """Run a simulated conversation and return output + session for evaluation."""
    # Reset booking state between cases so they don't contaminate each other
    global bookings, booking_counter
    bookings = {}
    booking_counter = 0

    simulator = ActorSimulator.from_case_for_user_simulator(case=case, max_turns=8)
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=ALL_TOOLS,
        trace_attributes={
            'gen_ai.conversation.id': case.session_id,
            'session.id': case.session_id,
        },
        callback_handler=None,
    )

    all_spans = []
    user_message = case.input
    agent_text = ''

    while simulator.has_next():
        memory_exporter.clear()
        agent_response = agent(user_message)
        agent_text = str(agent_response)
        all_spans.extend(list(memory_exporter.get_finished_spans()))
        user_result = simulator.act(agent_text)
        user_message = str(user_result.structured_output.message)

    mapper = StrandsInMemorySessionMapper()
    session = mapper.map_to_session(all_spans, session_id=case.session_id)
    return {'output': agent_text, 'trajectory': session}

print('Task function ready.')

## 5. Step 4a: Session-Level Binary Evaluation (`GoalSuccessRateEvaluator` in Assertion Mode)

`GoalSuccessRateEvaluator` has two modes:

| Mode | Trigger | Output |
|------|---------|--------|
| **Basic** | No `expected_assertion` on the case | Judge infers user goals from the conversation; returns Yes (1.0) or No (0.0) |
| **Assertion** | `expected_assertion` is set on the case | Judge evaluates against the assertion you wrote; returns SUCCESS (1.0) or FAILURE (0.0) |

Both modes are **already binary**. Assertion mode is strongly preferred for regression testing because the pass criterion is explicit and human-auditable instead of LLM-inferred. That is exactly the binary-custom-criterion pattern we want.

In [ ]:
# Step 4a: session-level binary evaluator
import textwrap

session_experiment = Experiment(
    cases=test_cases,
    evaluators=[GoalSuccessRateEvaluator(model=JUDGE_MODEL)],
)

print('Running GoalSuccessRateEvaluator (assertion mode) on all cases...')
print()
session_reports = session_experiment.run_evaluations(evaluation_task)

for report in session_reports:
    print(f'Evaluator: {report.evaluator_name}')
    print(f' Pass rate: {report.overall_score:.0%}')
    for i, (score, passed, reason) in enumerate(zip(report.scores, report.test_passes, report.reasons)):
        case_name = report.cases[i].get('name', f'Case {i}')
        verdict = 'PASS' if score >= 1.0 else 'FAIL'
        print(f' [{verdict}] {case_name}')
        wrapped = textwrap.fill(reason, width=96, initial_indent=' Reason: ', subsequent_indent=' ')
        print(wrapped)
    print()

## 6. Step 4b: Per-Turn / Per-Output Binary Evaluators (`OutputEvaluator` With Binary Rubrics)

For failure modes that live at the output or turn level (such as F1 *invented flight*, F6 *off-topic*, or F7 *wrong date format*), `OutputEvaluator` takes a natural-language **rubric** you write and has a judge LLM apply it. The rubric is the only place your domain expertise is encoded; everything else is plumbing.

To keep results binary, phrase the rubric as a single yes/no question and instruct the judge to treat the case as **PASS (1.0)** or **FAIL (0.0)**. The underlying `EvaluationOutput` carries a `test_pass` boolean, so you can use it directly without thresholding a float score.

We define three binary rubrics, one per targeted failure mode, and run them together as an experiment.

In [ ]:
# Step 4b: custom binary output evaluators, one per targeted failure mode.
#
# Each rubric is a single yes/no question phrased to produce a PASS/FAIL verdict.
# The judge is instructed to return score=1.0 for PASS and score=0.0 for FAIL.

RUBRIC_F1_NO_INVENTED_FLIGHTS = (
    'Binary check: does the agent reference ANY flight that was NOT returned by search_flights '
    'in this conversation (for example, a flight number or airline not in the search results)? '
    'Mark the case as PASS (score 1.0) if every flight mentioned by the agent appeared in search_flights output. '
    'Mark the case as FAIL (score 0.0) if the agent mentioned or booked any flight not present in search_flights output.'
)

RUBRIC_F6_NO_OFF_TOPIC = (
    'Binary check: did the agent answer an off-topic request (writing code, giving medical/legal advice, '
    'or any topic outside travel booking assistance)? '
    'Mark the case as PASS (score 1.0) if the agent stayed within travel booking assistance and politely '
    'redirected any off-topic request. '
    'Mark the case as FAIL (score 0.0) if the agent produced any off-topic content (code, unrelated advice, etc.).'
)

RUBRIC_F7_DATE_FORMAT = (
    'Binary check: did the agent invoke any booking tool (book_flight or book_hotel) with a date argument '
    'that is not in YYYY-MM-DD format, or with a date value that is obviously not a real calendar date? '
    'Mark the case as PASS (score 1.0) if every booking tool call used a valid YYYY-MM-DD date argument. '
    'Mark the case as FAIL (score 0.0) otherwise.'
)

# NOTE on architecture: Strands Evals' Experiment groups report results by the
# evaluator's class name. When multiple OutputEvaluator instances are passed
# to a single Experiment, their per-case results get merged under one key.
# To keep per-failure-mode reporting clean, we run each OutputEvaluator in
# its own Experiment below.

output_evaluator_specs = [
    ('F1_no_invented_flights', RUBRIC_F1_NO_INVENTED_FLIGHTS),
    ('F6_no_off_topic',        RUBRIC_F6_NO_OFF_TOPIC),
    ('F7_date_format',         RUBRIC_F7_DATE_FORMAT),
]

print('Running custom binary OutputEvaluators (one per failure mode)...')
print()

# Map label -> EvaluationReport so we can reference each by name later.
output_reports = {}
for label, rubric in output_evaluator_specs:
    exp = Experiment(
        cases=test_cases,
        evaluators=[OutputEvaluator(rubric=rubric, model=JUDGE_MODEL)],
    )
    reports = exp.run_evaluations(evaluation_task)
    output_reports[label] = reports[0]

for label, report in output_reports.items():
    print(f'[{label}] Pass rate: {report.overall_score:.0%}')
    for i, (score, passed) in enumerate(zip(report.scores, report.test_passes)):
        case_name = report.cases[i].get('name', f'Case {i}')
        verdict = 'PASS' if score >= 1.0 else 'FAIL'
        print(f' {verdict:4s} | {case_name}')
    print()

## 7. Step 5: Aggregate as Pass Rate Per Failure Mode

The point of building evaluators tied to failure modes is that you can read the evaluation results at a glance. Each column below is a failure mode. Each row is a case. A single FAIL stands out and tells you *what* broke and *where*, without comparing Likert scores across evaluators or wondering what the difference is between 0.5 and 0.75.

In [ ]:
# Combine session-level (GoalSuccessRate assertion) + output-level (binary rubrics) into
# a single pass/fail matrix indexed by case and failure mode.

# Column headers and the corresponding per-evaluator reports.
# output_reports is a dict keyed by failure-mode label (see previous cell).
columns = [
    ('GoalSuccess',          session_reports[0]),
    ('F1_invented_flights',  output_reports['F1_no_invented_flights']),
    ('F6_off_topic',         output_reports['F6_no_off_topic']),
    ('F7_date_format',       output_reports['F7_date_format']),
]

# Build per-case lookup: case_name -> { column_name: PASS/FAIL }
matrix = {}
case_names = [c.name for c in test_cases]
for col_name, report in columns:
    for i, cd in enumerate(report.cases):
        name = cd.get('name', f'Case {i}')
        matrix.setdefault(name, {})
        matrix[name][col_name] = 'PASS' if report.test_passes[i] else 'FAIL'

# Render
header = f'{"Case":35s} | ' + ' | '.join(f'{c:20s}' for c, _ in columns)
print(header)
print('-' * len(header))
for name in case_names:
    row = matrix.get(name, {})
    cells_str = ' | '.join(f'{row.get(col, "-"):20s}' for col, _ in columns)
    print(f'{name:35s} | {cells_str}')

print()
print('Pass rate per failure mode:')
for col_name, report in columns:
    print(f' {col_name:25s} {report.overall_score:6.0%}')

## 8. Step 6: Validate the LLM Judge Before Trusting Its Verdicts

An LLM judge is itself a component with failure modes. Before we treat its verdicts as ground truth, we should check that they agree with a human reviewer on a small labeled set.

The usual approach:

1. Run the pipeline on 20-30 cases.
2. A human labels each case PASS/FAIL for each evaluator.
3. Compare human labels vs judge labels and compute True Positive Rate (TPR) and True Negative Rate (TNR).
4. If TPR and TNR are both high (e.g., ≥ 0.9), trust the judge. If not, revise the rubric or swap the judge model.

The cell below shows the skeleton - it runs on a tiny 3-case subset for illustration. In practice you would scale this up and persist the human labels alongside your test cases.

In [ ]:
# Step 6: simple judge calibration skeleton.
# In practice, human_labels would come from a labeling UI or spreadsheet.

# For each case, a human has decided the expected PASS/FAIL for GoalSuccessRate.
# (Values below are the expected outcomes assuming the agent behaves correctly.)
human_labels_goal = {
    'full-booking-with-confirmation': True, # expect PASS
    'review-and-cancel-correct-ref': True, # expect PASS
    'off-topic-redirect': True, # expect PASS
}

# Pull the judge's verdicts for GoalSuccessRate from the session report we already have.
goal_report = session_reports[0]
judge_labels_goal = {}
for i, cd in enumerate(goal_report.cases):
    judge_labels_goal[cd.get('name', f'Case {i}')] = bool(goal_report.test_passes[i])

tp = fn = fp = tn = 0
for name, human_pass in human_labels_goal.items():
    judge_pass = judge_labels_goal.get(name, False)
    if human_pass and judge_pass: tp += 1
    elif human_pass and not judge_pass: fn += 1
    elif not human_pass and judge_pass: fp += 1
    else: tn += 1

tpr = tp / (tp + fn) if (tp + fn) else float('nan')
tnr = tn / (tn + fp) if (tn + fp) else float('nan')
print(f'Judge calibration vs human labels for GoalSuccessRate:')
print(f' True Positive Rate (sensitivity): {tpr:.2f}')
print(f' True Negative Rate (specificity): {tnr:.2f}')
print()
print(' If TPR or TNR is below ~0.9, revise the rubric/assertion or try a stronger judge model.')

## Next Steps

- Continue to **`04-11-06-synthetic-data.ipynb`** to generate scenarios along structured dimensions, then feed them back into this evaluator set.
- Or skip to **`04-11-08-e2e-pipeline.ipynb`** to see dimensions + simulation + custom binary evaluators wired into a single pipeline.